In [1]:
# !pip install torch torchvision onnx coremltools onnx-coreml
!pip install onnxsim

  Using cached onnxsim-0.4.36.tar.gz (21.0 MB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [11 lines of output]
      C:\Users\kenta\AppData\Local\Temp\pip-install-n8s7hkt0\onnxsim_c6397be0272a4528b2f1999d095202fd\setup.py:28: DeprecationWarning: Use shutil.which instead of find_executable
        CMAKE = find_executable('cmake')
      fatal: not a git repository (or any of the parent directories): .git
      fatal: not a git repository (or any of the parent directories): .git
      Traceback (most recent call last):
        File "<string>", line 2, in <module>
        File "<pip-setuptools-caller>", line 34, in <module>
        File "C:\Users\kenta\AppData\Local\Temp\pip-install-n8s7hkt0\onnxsim_c6397be0272a4528b2f1999d095202fd\setup.py", line 67, in <module>
          assert CMAKE, 'Could not find "cmake" executable!'
                 ^^^^^
      AssertionError: Could not find "cmake" executable!
      [end of output]
  
  note: This error originates from 

In [ ]:
import torch

model = torch.hub.load(
    'facebookresearch/detr:main',
    'detr_resnet50',
    pretrained=True
)
model.eval()

dummy_input = torch.randn(1, 3, 800, 1333)
torch.onnx.export(
    model,
    dummy_input,
    "model.onnx",
    opset_version=13,
    input_names=["input_image"],
    output_names=["pred_logits", "pred_boxes"],
    dynamic_axes={
        "input_image": {0:"batch_size"},
        "pred_logits": {0:"batch_size"},
        "pred_boxes": {0:"batch_size"}
    }
)

import onnx
from onnxsim import simplify

onnx_model = onnx.load("model.onnx")
onnx.checker.check_model(onnx_model)  # モデル整合性チェック
simplified_model, check = simplify(onnx_model)
assert check, "ONNX simplifyに失敗しました"
onnx.save(simplified_model, "model_simplified.onnx")


import coremltools as ct

mlmodel = ct.converters.onnx.convert(
    model="model_simplified.onnx",
    minimum_deployment_target=ct.target.iOS14,
    image_input_names=["input_image"],
    image_scale=1/255.0,
    compute_units=ct.ComputeUnit.ALL
)
mlmodel.save("ObjectDet.mlmodel")
